In [1]:
import numpy as np

In [31]:
import numpy as np

def calculate_mesh_deformation_gradient(node_coords, cells, node_disp):
    """
    Calculates the constant deformation gradient F for every cell in a 2D triangular mesh.
    
    Args:
        node_coords (np.ndarray): (N, 2) array of nodal X, Y coordinates.
        cells (np.ndarray): (M, 3) array of node indices for M triangles.
        node_disp (np.ndarray): (N, 2) array of nodal u, v displacements.
        
    Returns:
        np.ndarray: (M, 2, 2) array of deformation gradient F matrices.
    """
    num_cells = cells.shape[0]
    F_matrices = np.zeros((num_cells, 2, 2))  # Initialize output (M x 2 x 2)

    for i, cell_nodes in enumerate(cells):
        # --- A. Extract Data for the current cell ---
        
        # Undeformed coordinates of the 3 nodes: (3, 2) array
        cell_coords = node_coords[cell_nodes, :]  
        
        # Displacements of the 3 nodes: (3, 2) array. Flattened to (6,) for calculation.
        cell_disps_2d = node_disp[cell_nodes, :]
        cell_disps = cell_disps_2d.flatten()  # [u1, v1, u2, v2, u3, v3]

        # --- B. Calculation Steps (based on CST formulation) ---

        x = cell_coords[:, 0]  # X coordinates [X1, X2, X3]
        y = cell_coords[:, 1]  # Y coordinates [Y1, Y2, Y3]
        
        u_nodes = cell_disps[0::2]  # [u1, u2, u3]
        v_nodes = cell_disps[1::2]  # [v1, v2, v3]

        # 1. Area (Ae) - Essential for normalization
        Ae = 0.5 * np.linalg.det(np.array([
            [1.0, x[0], y[0]],
            [1.0, x[1], y[1]],
            [1.0, x[2], y[2]]
        ]))
        
        # 2. Geometric Constants (Beta and Gamma)
        # Beta = d(N_i)/d(X); Gamma = d(N_i)/d(Y) (multiplied by 2*Ae)
        beta = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]]) 
        gamma = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]]) 
        
        # 3. Displacement Gradient (H) components
        # H_IJ = d(I)/d(J) (where I is displacement component, J is undeformed coordinate)
        
        # H_XX = d(u)/d(X)
        H_XX = np.dot(u_nodes, beta) / (2 * Ae)
        # H_XY = d(u)/d(Y)
        H_XY = np.dot(u_nodes, gamma) / (2 * Ae)
        
        # H_YX = d(v)/d(X)
        H_YX = np.dot(v_nodes, beta) / (2 * Ae)
        # H_YY = d(v)/d(Y)
        H_YY = np.dot(v_nodes, gamma) / (2 * Ae)
        
        # 4. Assemble H and F
        H = np.array([[H_XX, H_XY], [H_YX, H_YY]])
        F = np.eye(2) + H
        
        # --- C. Store the result ---
        F_matrices[i, :, :] = F

    return F_matrices

# Example of dummy data setup:
# N=4 nodes, M=2 cells (forming a square)
node_coords_example = np.array([[0., 0.], [1., 0.], [1., 1.], [0., 1.]])
cells_example = np.array([[0, 1, 2], [0, 2, 3]]) # Two triangles: (0,1,2) and (0,2,3)
# Displacement simulating a simple tension in X (uniform strain)
node_disp_example = np.array([[0.0, 0.0], [0.1, 0.0], [0.1, 0.0], [0.0, 0.0]])

# Calculate F for the mesh
# F_results = calculate_mesh_deformation_gradient(node_coords_example, cells_example, node_disp_example)

# print("Deformation Gradients (F) for all cells:\n", F_results)

In [33]:
data = np.load("/home/mmdiscovery/shared/dataset/NH/load0.5.npz")
mesh_pos = data["mesh_pos"]
cells = data["cells"]
u = data["u"]
# F_results = calculate_mesh_deformation_gradient(node_coords, cells, u)

In [11]:
F_results

array([[[ 1.24470589e+00,  3.85613329e-05],
        [ 0.00000000e+00,  1.51283742e+00]],

       [[ 1.24466733e+00,  0.00000000e+00],
        [ 1.09594724e-04,  1.51294701e+00]],

       [[ 1.24471690e+00,  3.45317931e-05],
        [ 2.64135415e-04,  1.51274074e+00]],

       ...,

       [[ 1.25379697e+00, -5.41360226e-05],
        [ 3.07532326e-05,  1.50017833e+00]],

       [[ 1.25376379e+00, -1.33616941e-05],
        [ 0.00000000e+00,  1.50020571e+00]],

       [[ 1.25377716e+00,  0.00000000e+00],
        [ 1.51757126e-05,  1.50022089e+00]]])

In [8]:
import torch
import numpy as np

In [9]:
def compute_basis(xi_eta):
    # For a 3-node linear triangle in 2D (xi, eta)
    N1 = 1.0 - xi_eta[:, 0] - xi_eta[:, 1]
    N2 = xi_eta[:, 0]
    N3 = xi_eta[:, 1]
    N = torch.stack([N1, N2, N3], dim=1) # (N_qp, 3)

    # Derivatives w.r.t. parent coordinates (xi, eta)
    dN_dxi = torch.tensor([[-1.0, 1.0, 0.0],
                           [-1.0, 0.0, 1.0]], dtype = xi_eta.dtype, device=xi_eta.device) # (2, 3)
    dN_dxi = dN_dxi.unsqueeze(0).expand(xi_eta.shape[0], -1, -1) # (N_qp, 2, 3)
    return N, dN_dxi

In [28]:

# Define material parameters (use actual values from your problem)
MU = 0.5  # Shear Modulus (mu/2 factor is in the formula)
K_BULK = 1.5  # Bulk Modulus (K/2 factor is in the formula)

def NH_model(F: torch.Tensor, mu: float = MU, K: float = K_BULK) -> torch.Tensor:
    """
    Calculates the total Strain Energy (integral approximation) for a batch of Deformation Gradients.
    
    Args:
        F (torch.Tensor): Batch of Deformation Gradients (N_qp, D, D).
        mu (float): Shear Modulus.
        K (float): Bulk Modulus.
        
    Returns:
        torch.Tensor: The sum of the energy density over all quadrature points (a single scalar).
    """
    
    # Ensure F has gradient tracking enabled if solving for F
    
    # 1. Kinematics
    J = torch.det(F) # (N_qp,)
    C = F.transpose(-2, -1) @ F # (N_qp, D, D)
    
    # 2. Invariants
    # I1 = tr(C) - The trace operates on the last two dimensions
    I1 = torch.einsum('...ii->...', C) # (N_qp,)

    # 3. Isochoric and Volumetric Parts (Your formula uses mu/2 and K/2 factors implicitly)
    # Your formula: Psi = 0.5 * (I1_bar - 3) + 1.5 * (J - 1)^2
    # Where I1_bar = J**(-2/3) * I1
    
    # Isochoric Energy: Psi_iso = (mu/2) * (I1_bar - 3)
    I1_bar = torch.pow(J, -2/3) * I1
    Psi_iso = (mu / 2.0) * (I1_bar - 3.0)

    # Volumetric Energy: Psi_vol = (K/2) * (J - 1)^2
    Psi_vol = (K / 2.0) * torch.pow(J - 1.0, 2)
    
    # Total Energy Density: (N_qp,)
    Psi = Psi_iso + Psi_vol
    
    # 4. Critical Step: Return a single scalar (the sum/approximation of the integral)
    # This allows autograd to compute the gradient (P_e) for all N_qp simultaneously.
    return Psi.sum()

In [29]:
def compute_weak_form_residual(node_coords, connectivity, u):
    device = node_coords.device
    N_elem = connectivity.shape[0]
    N_vtx = connectivity.shape[1] # 3 for triangles
    D = node_coords.shape[1] - 1      # 2 for 2D

    # --- Gauss Quadrature Setup (1-point integration for simplicity) ---
    # Weight and location in parent coordinates (xi, eta)
    # qp_coords = torch.tensor([[1/3, 1/3]], dtype=node_coords.dtype, device=device) # (1, 2)
    # qp_weights = torch.tensor([0.5], dtype=node_coords.dtype, device=device)       # (1,)
    qp_coords = torch.tensor([
    [0.5, 0.0],  # Midpoint of edge 2-3
    [0.5, 0.5],  # Midpoint of edge 3-1
    [0.0, 0.5]   # Midpoint of edge 1-2
        ], dtype=node_coords.dtype, device=device)

    qp_weights = torch.tensor([1/3.0, 1/3.0, 1/3.0], 
                                dtype=node_coords.dtype, device=device)
    N_qp = qp_coords.shape[0]

    # Compute basis functions and their derivatives at quadrature points
    N, dN_dxi = compute_basis(qp_coords) # N: (1, 3), dN_dxi: (1, 2, 3)

    # Initialize the Residual Vector (the weak form residual)
    # The residual is typically a force vector R, R_i = Integral(...) * v_i
    R_internal = torch.zeros_like(u, dtype = node_coords.dtype) # (N_nodes, D)

    # --- Element Loop ---
    for e in range(N_elem):
        # 1. Get Nodal Information for the current element
        node_indices = connectivity[e]
        X_e = node_coords[node_indices][:, :2] # (3, 2)
        u_e = u[node_indices][:, :2]           # (3, 2)

        # 2. Compute Jacobian Matrix and its determinant
        # J_ij = dX_i / d(xi_j). J = dX/d(xi)
        # J_2x2 = dN/d(xi) * X_e (sum over N_vtx)
        J_mat = dN_dxi @ X_e # (N_qp, 2, 2)
        det_J = torch.det(J_mat).unsqueeze(-1) # (N_qp, 1)

        # 3. Compute the Jacobian Inverse and the Physical Gradient of N
        J_inv = torch.inverse(J_mat) # (N_qp, 2, 2)
        # dN/dX = J_inv * dN/d(xi) 
        dN_dX = J_inv @ dN_dxi # (N_qp, 2, 3) -> dN_dX_ij = dN_j / dX_i

        # 4. Compute Deformation Gradient F_e and PK1 Stress P_e
        # F = I + d(u)/dX. d(u)/dX = d(u_e)/dX = dN/dX * u_e (u_e is 3x2, dN/dX is 2x3)
        du_dX = dN_dX @ u_e # (N_qp, 2, 2)
        F_e = torch.eye(D, device=device) + du_dX # (N_qp, 2, 2)
        
        # Calculate P_e: The PK1 Stress Tensor
        P_e = torch.autograd.grad(NH_model(F_e), F_e)[0] # (N_qp, 2, 2)
        
        # 5. Compute the Internal Virtual Work Residual $\int_{\Omega_E} \mathbf{P} : \nabla(\delta\mathbf{u}) \, dV$
        # The virtual strain measure is $ \nabla(\delta\mathbf{u}) $. 
        # In FEM, the test function $\delta\mathbf{u}$ is $\sum_a N_a \delta\mathbf{u}_a$.
        # $\nabla(\delta\mathbf{u}) = \sum_a \nabla N_a \delta\mathbf{u}_a$.
        # The contraction must be $\mathbf{P} : (\sum_a \nabla N_a \delta\mathbf{u}_a)$.
        # The elemental residual vector $\mathbf{R}_a$ (force at node $a$) is the coefficient of $\delta\mathbf{u}_a$:
        # $\mathbf{R}_a = \int_{\Omega_E} \mathbf{P} \cdot \nabla N_a \, dV$

        # R_a (vector, D) = sum_qp (P_e * dN_dX[:, :, a].T) * det_J * w_qp
        R_e = torch.zeros(N_vtx, D, device=device, dtype = node_coords.dtype) # (3, 2)
        
        # Loop over nodes 'a' for the shape function $N_a$
        for a in range(N_vtx):
            # Gradient of N_a: dN/dX_a (2, 1) at all qp
            dN_dX_a = dN_dX[:, :, a].unsqueeze(-1) # (N_qp, 2, 1)

            # Contraction: P_e @ dN_dX_a (The result is a force vector per area, 2x1)
            # P_ij * dN_j/dX_i (implied sum over j)
            Force_per_Area = P_e @ dN_dX_a # (N_qp, 2, 1)

            # Quadrature Sum: Integral = Sum(F/A * det_J * w_qp)
            # Force_per_Area is 2x1, det_J is 1x1, w_qp is 1x1
            # We use broadcasting and then sum over quadrature points (dim 0)
            Integral_Term = Force_per_Area * det_J.unsqueeze(-1) * qp_weights.view(-1, 1, 1)
            
            R_e[a] = Integral_Term.sum(dim=0).squeeze(-1) # (2,)

        # 6. Assemble into the global residual vector
        # R_internal[node_indices] += R_e (This is the standard FEM assembly step)
        # PyTorch assembly needs scatter_add or a direct indexing equivalent.
        # print(R_e.dtype)
        # print(R_internal.dtype)
        R_internal.scatter_add_(0, node_indices.unsqueeze(1).expand(-1, D), R_e)
        
    return R_internal


In [30]:
data = np.load("/home/mmdiscovery/shared/dataset/NH/load0.5.npz")
mesh_pos = data["mesh_pos"]
cells = data["cells"]
u = data["u"]
nt = data["node_type"]

In [31]:
mesh_pos = torch.tensor(mesh_pos, requires_grad=True)
mesh_pos.to(torch.float64)
cells = torch.tensor(cells, dtype = torch.int64) 
u = torch.tensor(u, requires_grad=True)
u.to(torch.float64)
nt = torch.tensor(nt)

In [32]:
res = compute_weak_form_residual(mesh_pos, cells, u)

In [33]:
res

tensor([[-1.0375e-01,  9.3378e-02],
        [-5.4816e-06,  1.1841e-05],
        [ 5.8858e-06,  1.8674e-01],
        ...,
        [-6.6037e-06, -7.6944e-06],
        [ 2.0603e-01, -6.4111e-06],
        [ 1.0301e-01, -9.3516e-02]], dtype=torch.float64,
       grad_fn=<ScatterAddBackward0>)

In [34]:
float(torch.mean(torch.norm(res[nt == 0], p = 2, dim = 1)).detach())

0.0006257502641188417

In [137]:
mesh_pos.shape

torch.Size([521, 3])

In [101]:
nt == 0

array([False,  True, False, False,  True,  True, False, False,  True,
       False,  True,  True,  True,  True, False, False,  True, False,
        True,  True,  True,  True,  True,  True, False,  True, False,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True, False, False,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False, False,
        True,  True, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False, False,
        True,  True, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False, False,  True,  True,  True,  True,
       False, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,